# 03 – Rurality: do ambulances take longer where people are more spread out?

**Where we are:** in notebook 02 we found a surprise. The *most deprived* trust areas get the *fastest* Category 2 responses.
We noticed the slowest trusts (East Midlands, East of England, South Western) cover large rural areas, so we guessed that
**travel distance** might be the real reason.

**This notebook tests that idea** using population density (people per square kilometre):
1. Load ONS population and land area for every council area
2. Link them to ambulance trusts using our lookup table from notebook 02
3. Build three measures of how spread out each trust's population is
4. Compare them with Category 1 and Category 2 response times
5. Compare rurality against deprivation: which is the stronger link?

**Data source:** Office for National Statistics (ONS), *Estimates of the population for the UK, England, Wales, Scotland and
Northern Ireland*, mid-2025 edition, table **MYE5** (population density for local authorities). Public, aggregated data.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_DIR = Path("..")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIG_DIR = PROJECT_DIR / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the ONS population density table

The Excel file has many sheets. We need **MYE5**.

Like many official spreadsheets, the sheet starts with a few rows of notes before the real table.
Instead of guessing how many rows to skip, we find the row whose first cell says `Code` and use that as the header.
This keeps working if ONS adds or removes a note line next year.

In [ ]:
ons_files = sorted(RAW_DIR.glob("mye*tables*.xlsx"))
assert ons_files, f"No ONS population estimates file (mye...tables...xlsx) found in {RAW_DIR.resolve()}"
ons_path = ons_files[-1]
print("Loading:", ons_path.name)

preview = pd.read_excel(ons_path, sheet_name="MYE5", header=None, usecols=[0])
header_row = preview.index[preview[0] == "Code"][0]
print("Table header is on row", header_row)

mye5 = pd.read_excel(ons_path, sheet_name="MYE5", header=header_row)
print("Rows, columns:", mye5.shape)
mye5.head()

The table has a population and density column for every year from 2011. We only need the **latest year**,
and we pick it out automatically so the code still works with next year's file.

In [ ]:
pop_cols = [c for c in mye5.columns if str(c).startswith("Estimated Population mid-")]
latest_pop_col = max(pop_cols, key=lambda c: int(str(c)[-4:]))
latest_year = latest_pop_col[-4:]
print("Latest year:", latest_year)

density = mye5[["Code", "Name", "Geography", "Area (sq km)", latest_pop_col]].rename(columns={
    "Code": "area_code",
    "Area (sq km)": "area_km2",
    latest_pop_col: "population",
})
density["people_per_km2"] = density["population"] / density["area_km2"]
density["Geography"].value_counts()

## 3. Link to ambulance trusts

The table contains every level of geography (countries, regions, counties, districts...).
We only keep the 153 council areas in our lookup table, matching on the official **area code** (e.g. `E06000001`),
not the name. Names are written differently in different datasets (e.g. "Bristol" vs "Bristol, City of"), but codes are not.

In [ ]:
lookup = pd.read_csv(PROCESSED_DIR / "council_area_to_ambulance_trust_lookup.csv")

areas = lookup.merge(density, on="area_code", how="left", validate="one_to_one")

missing = areas[areas["population"].isna()]
print("Council areas with no ONS match:", len(missing))
assert missing.empty, missing

print("Geography types matched:")
areas["Geography"].value_counts()

In [ ]:
# Sense check: total population and land area of England
print(f"Population (mid-{latest_year}): {areas['population'].sum():,.0f}")
print(f"Land area: {areas['area_km2'].sum():,.0f} sq km")

In [ ]:
# The most and least densely populated council areas
areas.sort_values("people_per_km2")[["area_name", "trust", "people_per_km2"]].iloc[[0, 1, 2, -3, -2, -1]].round(0)

## 4. Three ways to measure how spread out a trust's population is

There isn't one perfect measure, so we try three and see if they agree.

**a) Simple density** = total population ÷ total land area.
Easy, but misleading: the North East has empty moorland in Northumberland, but most people live in Newcastle and Sunderland.

**b) Population-weighted density** = the density of the area where the *average person* lives.
Each council area's density is weighted by its population. This better reflects where 999 calls actually come from.

**c) Land area per 1,000 Category 2 calls** = how much ground the trust covers for its workload.
A trust with lots of land but few calls has to spread its ambulances thinly.

In [ ]:
areas["pop_x_density"] = areas["population"] * areas["people_per_km2"]

trust_density = areas.groupby("trust").agg(
    land_km2=("area_km2", "sum"),
    population_ons=("population", "sum"),
    pop_x_density=("pop_x_density", "sum"),
)
trust_density["simple_density"] = trust_density["population_ons"] / trust_density["land_km2"]
trust_density["pop_weighted_density"] = trust_density["pop_x_density"] / trust_density["population_ons"]
trust_density = trust_density.drop(columns="pop_x_density")
trust_density.round(0).sort_values("pop_weighted_density")

Now we join this to the trust table from notebook 02 (response times + deprivation, age and mortality).

In [ ]:
trust = pd.read_csv(PROCESSED_DIR / "trust_response_and_area_measures.csv").set_index("trust")
trust = trust.join(trust_density, how="inner", validate="one_to_one")
trust["km2_per_1000_c2_calls"] = trust["land_km2"] / trust["c2_calls"] * 1000

print("Trusts:", len(trust))
trust[["simple_density", "pop_weighted_density", "km2_per_1000_c2_calls",
       "imd_score", "c1_mean_min", "c2_mean_min"]].round(1).sort_values("pop_weighted_density")

## 5. Does rurality line up with response times?

We look at **both Category 1 and Category 2**:
- **Category 1** (life-threatening) gets the nearest ambulance sent straight away, so it mostly depends on **travel distance**.
- **Category 2** calls often wait in a queue when ambulances are busy (for example, stuck outside hospitals waiting to hand over patients),
  so they depend more on **how stretched the service is**.

If rurality matters, we'd expect it to show up most clearly for Category 1.

We use a **log scale** on the density axis because London is about 5 times denser than any other trust and would squash everyone else into a corner.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, cat, label in [(axes[0], "c1_mean_min", "Category 1"), (axes[1], "c2_mean_min", "Category 2")]:
    rho = trust["pop_weighted_density"].corr(trust[cat], method="spearman")
    ax.scatter(trust["pop_weighted_density"], trust[cat])
    for name, row in trust.iterrows():
        offset = (4, -10) if name == "East of England" else (4, 3)
        ax.annotate(name, (row["pop_weighted_density"], row[cat]), fontsize=8,
                    xytext=offset, textcoords="offset points")
    ax.set_xscale("log")
    ax.set_xticks([400, 700, 1000, 2000, 4000, 7000])
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.xaxis.set_minor_formatter(FuncFormatter(lambda x, _: ""))
    ax.set_xlabel("Population-weighted density (people per sq km, log scale)\n← more rural        more urban →")
    ax.set_ylabel(f"{label} mean response (minutes)")
    ax.set_title(f"{label}: Spearman correlation = {rho:.2f}")

fig.suptitle(f"Response times vs how densely populated each trust's area is (12 months to Sep 2026)")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_density_vs_response.png", dpi=150)
plt.show()

## 6. Rurality vs deprivation: which is the stronger link?

Now we put every measure side by side, for both categories.
For the rurality measures, a **negative** number for density means *more rural → slower*,
and a **positive** number for land per call also means *more rural → slower*.

We also re-run without **London**, which is very dense but still slow, to check whether one unusual trust is driving the results.

In [ ]:
measures = {
    "simple_density":        "Simple density",
    "pop_weighted_density":  "Population-weighted density",
    "km2_per_1000_c2_calls": "Land area per 1,000 C2 calls",
    "imd_score":             "Deprivation (IMD 2019)",
    "pct_65_plus":           "% aged 65+",
    "c2_calls_per_1000":     "C2 calls per 1,000 people",
}

def spearman_table(data):
    return pd.DataFrame({
        "Category 1": {label: data[col].corr(data["c1_mean_min"], method="spearman") for col, label in measures.items()},
        "Category 2": {label: data[col].corr(data["c2_mean_min"], method="spearman") for col, label in measures.items()},
    })

corr_all = spearman_table(trust)
corr_no_london = spearman_table(trust.drop(index="London")).add_suffix(" (no London)")
corr_summary = corr_all.join(corr_no_london).round(2)
corr_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
corr_all.plot.barh(ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-1, 1)
ax.set_xlabel("Spearman correlation with mean response time")
ax.set_title("How strongly does each measure line up with response times? (11 trusts)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / "03_correlation_summary.png", dpi=150)
plt.show()

### Are rurality and deprivation measuring the same thing?
If urban trusts were also the more deprived ones, we couldn't easily tell the two effects apart.
Let's check how they relate to each other (positive = denser trusts are more deprived).

In [ ]:
trust[["pop_weighted_density", "km2_per_1000_c2_calls", "imd_score"]].corr(method="spearman").round(2)

## 7. Save

In [ ]:
out = PROCESSED_DIR / "trust_response_area_and_density.csv"
trust.reset_index(names="trust").to_csv(out, index=False)
corr_summary.to_csv(PROCESSED_DIR / "correlation_summary.csv")
print("Saved", out)

## 8. Summary

**Our idea was only partly right.**

- **Rurality matters most for Category 1.** Where people live more spread out, life-threatening calls take longer to reach
  (population-weighted density vs C1: −0.64). This makes sense: Category 1 is mainly about travel distance.
- **For Category 2, the link with rurality is weaker** (−0.43). The North East shows why: it covers lots of rural land,
  yet it has the fastest Category 2 times in England.
- **London changes the picture.** It is by far the densest trust but is still slow for Category 2. Without London,
  the Category 2 link with population-weighted density rises to about −0.6, a sign that London's delays have other causes
  (for example, traffic or waiting to hand patients over at busy hospitals).
- **Deprivation is still the strongest single link**, in the "surprising" direction: more deprived trusts are faster.
- **Rurality and deprivation partly overlap.** Trusts where people live in denser areas also tend to be more deprived
  (correlation about 0.45). So some of the "deprived areas are faster" pattern may really be "urban areas are faster".
  With only 11 trusts, we can't fully separate the two.

**What this suggests:** distance clearly matters for the most urgent calls, but it doesn't explain everything.
The remaining differences, especially for Category 2, probably depend on **how each service and its local hospitals are
run and resourced** (staffing, hospital handover delays), which this data doesn't cover.

**Limitations:**
- Only 11 trusts, so these correlations are clues, not proof.
- Correlation is not causation.
- Density by council area is a rough measure of rurality. Large counties contain both towns and countryside.
- No data yet on staffing or hospital handover delays.